# SOFR futures convexity adjustment vs USD SOFR butterflies — the fresh backtest

> *"Convexity adjustments for 1y SOFR packs are computed as the spread between
> the pack's rate (the average of 4 SOFR rates in the pack) and
> matched-maturity forward 1y CME swap rate."* — Citi Research, Rates Vol Lab,
> 12-Jun-2023 (Fig 58, the SOFR restatement of the Eurodollar screen)

> *"Sell $200k DV01 of Blues CAs … pay the belly of the 2s5s10s swap fly with
> notional weights $147mm/−$85.6mm/$20.89mm (0.705/−1/0.465 DV01 weights)."*
> — Citi, 09-Feb-2017, the ticket this family of trades descends from

This block re-measures the whole family from scratch — **outrights, packs and
CME bundles against spot AND forward-starting butterflies, five signal
families, 515 pre-registered cells** — on the repaired Q/Q CA path, trusting
none of the previous results. The pre-registration
(`docs/convexityrv/cavf-grid-preregistration.md`) was frozen before scoring
and carries two dated execution amendments, both discovered by this block's
own machinery:

1. **Same-day fills harvested the CA mark's own measurement noise** — the
   first pass printed 100% hit rates over 3–5-day holds. Primary convention
   is now fills at t+1; the same-day gap is reported as the noise harvest.
2. **Constant-rank labels booked the IMM-roll contract switch as P&L** —
   22 of 33 SR3 rolls are FOMC dates, so the jump is systematically signed.
   Signals and panel P&L now use roll-spliced series.

**The verdict is negative and it survives both corrections being generous.**
Every CA-based family's MEDIAN gross is below zero before costs; the single
best cell of 515 sits below the annualised E[max SR | null]; and the only
family with a positive median gross is the fly-only CONTROL — which is not a
convexity trade and dies on its own costs. Sections 7–9 carry the numbers.

One interpretive note the corpus adds (Huggins & Schaller 2022, ch. 6): the
SR3 contract's payout lands at the END of its reference period, so it has
**no Jensen convexity of its own** — the measured CA is financing/margin bias
plus positioning, which is exactly why the fundamental overlays (CFTC dealer
positioning, CME–LCH basis) were declared. They did not rescue the trade
either (§7).

In [1]:
import nest_asyncio
nest_asyncio.apply()

import datetime as dt
import json
import math
import os
import pathlib
import sys

os.environ.setdefault("ARBS_SUPABASE_ENABLED", "0")
REPO = pathlib.Path.cwd()
while not (REPO / "RVUtils").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import statsmodels.api as sm

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"

from RVUtils.ConvexityRV import cavf_grid as G
from RVUtils.ConvexityRV import cavf_signals as S
from RVUtils.ConvexityRV import cavf_universe as U
from RVUtils.ConvexityRV import strat2_fly_universe as FU

DATA = REPO / "notebooks" / "data" / "convexity_rv"
pd.set_option("display.width", 220, "display.max_columns", 40)
print(f"repo {REPO}")

C:\Users\chris\clee\ARBS-cvx3\RVUtils\ConvexityRV\curve_ops.py:61: LicenceNotice:


Rateslib is source-available (not open-source) software distributed under a dual-licence model.
No commercial licence is registered for this installation. Use is therefore permitted for non-commercial purposes only (at-home or university based academic use).
Any use in commercial, professional, or for-profit environments, including evaluation or trial use, requires a valid commercial licence or an approved evaluation licence.
Certain features may require a registered commercial or evaluation licence in current or future versions.
For licensing information or to register a licence, please visit: https://rateslib.com/licence



repo C:\Users\chris\clee\ARBS-cvx3


## 1. CONFIG — every knob, and why it is set where it is

The config mirrors the pre-registration; the assert below refuses to present
numbers produced under a different one. Change a knob → re-run
`_cavf_run_grid.py` → the meta file moves with it.

In [2]:
import dataclasses


@dataclasses.dataclass(frozen=True)
class CavfConfig:
    #: signal/z window, rows. The CA panel measured 100% dense, so rows≈days.
    window: int = 252
    #: pre-registered primaries; the 1.5 set is the declared sensitivity.
    z_entry: float = 2.0
    z_exit: float = 0.5
    max_hold_bd: int = 63
    #: fills lag decisions by one mark (amendment 1). 0 = the noise diagnostic.
    exec_lag_bd: int = 1
    #: β gate, bp per bp (Citi's 21.4 in bp-per-percent ≡ 0.214 here).
    beta_abs_min: float = 0.01
    beta_abs_max: float = 1.0
    #: per-leg round-trip costs on leg DV01 (bp): futures package / swap / fly leg.
    cost_fut_bp: float = 0.25
    cost_swap_bp: float = 0.5
    cost_fly_leg_bp: float = 0.5
    #: the declared trial count. Moving it without amending the
    #: pre-registration is the trial-count leak the tests refuse.
    declared_trials: int = 515
    ca_dv01: float = 100_000.0


CFG = CavfConfig()
META = json.loads((DATA / "cavf_grid_meta.json").read_text())
assert META["declared_trials"] == CFG.declared_trials, (
    f"grid ran {META['declared_trials']} trials, config declares "
    f"{CFG.declared_trials} — the pre-registration and the artifacts disagree")
assert META["exec_lag_bd"] == CFG.exec_lag_bd
assert META["roll_spliced"] is True
print(json.dumps({k: v for k, v in dataclasses.asdict(CFG).items()}, indent=1))
print(f"\ngrid ran at {META['ran_at']}  |  {META['n_imm_rolls']} IMM rolls in window")

{
 "window": 252,
 "z_entry": 2.0,
 "z_exit": 0.5,
 "max_hold_bd": 63,
 "exec_lag_bd": 1,
 "beta_abs_min": 0.01,
 "beta_abs_max": 1.0,
 "cost_fut_bp": 0.25,
 "cost_swap_bp": 0.5,
 "cost_fly_leg_bp": 0.5,
 "declared_trials": 515,
 "ca_dv01": 100000.0
}

grid ran at 2026-08-24T08:23:06  |  22 IMM rolls in window


## 2. The panels

CA through the repaired TB path (`sfr_cvx_adj`, Q/Q matched swap, no
magnitude heuristics, `fill=False`), 31 labels × 1,409 dates, **zero pricing
failures**; fly legs at 48 tenors (spot + 1/2/3/4/5y forward starts), 100%
dense. The term structure must be monotone in rank — CA is a variance
quantity — and that is asserted, not eyeballed.

In [3]:
ca_wide = pd.read_parquet(DATA / "cavf_ca_panel.parquet")
ca_wide.index = pd.to_datetime(ca_wide.index)
cols = {c.split()[1]: c for c in ca_wide.columns}
failures = json.loads((DATA / "cavf_ca_failures.json").read_text())
assert not failures, f"the CA backfill recorded failures: {failures}"

ROLLS = S.imm_roll_dates(ca_wide.index)
ca_raw = {lab: ca_wide[c].dropna() for lab, c in cols.items()}
ca_spl = {lab: S.roll_splice(s, ROLLS) for lab, s in ca_raw.items()}

COLOURS = ["WHITES", "REDS", "GREENS", "BLUES", "GOLDS"]
med = pd.Series({c: float(ca_raw[c].median()) for c in COLOURS})
print("median CA by colour, bp:")
print(med.round(3).to_string())
assert med.is_monotonic_increasing, (
    "the adjustment is not increasing with rank — a data problem, not a view")

legs = pd.read_parquet(DATA / "cavf_fly_legs.parquet")
wide = FU.legs_wide(legs)
wide.index = pd.to_datetime(wide.index)
FLIES = {f.fly_id: f for f in U.tradeable_fly_specs()}
fly_by_id = {fid: FU.fly_rate_series(wide, f, 0.5, 0.5).dropna()
             for fid, f in FLIES.items()}
print(f"\nCA {ca_wide.shape}  legs {wide.shape}  flies {len(fly_by_id)}  "
      f"rolls {len(ROLLS)}")
neg = pd.Series(META["neg_ca_frac"])
print(f"negative-CA fraction: front noise (WHITES {neg['WHITES']:.2f}, "
      f"SFR2 {neg['SFR2']:.2f}) vs deep (GOLDS {neg['GOLDS']:.2f}) — the front "
      "prints negative on quote noise alone, as measured in previous blocks")

median CA by colour, bp:
WHITES     0.238
REDS       0.948
GREENS     4.224
BLUES      8.712
GOLDS     13.185

CA (1409, 31)  legs (1407, 48)  flies 18  rolls 22
negative-CA fraction: front noise (WHITES 0.38, SFR2 0.41) vs deep (GOLDS 0.01) — the front prints negative on quote noise alone, as measured in previous blocks


## 3. Does the machine give the right answer to questions we already know?

### 3.1 Citi's published SOFR screen, close 6/9/2023

Our BLUES and GOLDS on 2023-06-09 are the same four-contract windows as
Citi's printed M6-H7 and M7-H8 rows. The bar is **bp on a level** — the
correlation-only version of this tie-out was shown to survive ×2, ×100 and
+10bp mutations, i.e. to grade nothing. The annual-frequency matched swap is
the negative control and must FAIL by around −4bp.

In [4]:
CITI_FIG58 = {"BLUES": 15.40, "GOLDS": 22.29}   # Citi, close 6/9/2023
d0 = pd.Timestamp("2023-06-09")
ours = {lab: float(ca_raw[lab].loc[d0]) for lab in CITI_FIG58}
for lab, printed in CITI_FIG58.items():
    err = ours[lab] - printed
    print(f"{lab}: ours {ours[lab]:6.2f}  Citi {printed:6.2f}  err {err:+.2f}bp")
    assert abs(err) < 1.6, (
        f"{lab} misses Citi's printed screen by {err:+.2f}bp — the shared-path "
        "tie-out holds to ~1bp and this panel should too")

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.IRSwapsTB import IRSwapsTB

_tb = IRSwapsTB(IRSwapsMDP(source="citivelo_excel_rl"), show_tqdm=False,
                use_ts_cache=False)
_annual = _tb.sfr_cvx_adj(["BLUES"], dt.date(2023, 6, 9), dt.date(2023, 6, 9),
                          matched_frequency=None, matched_leg2_frequency=None)
_tb.close()
ann_val = float(_annual.iloc[0, 0])
gap = ann_val - ours["BLUES"]
print(f"\nnegative control — annual matched swap: {ann_val:.2f}bp "
      f"(gap {gap:+.2f}bp vs Q/Q)")
assert gap < -3.0, (
    "the annual-frequency negative control no longer fails — the Q/Q "
    "convention has been lost somewhere in the path")
print("OK: the annual convention fails by the documented ~4bp, as it must.")

BLUES: ours  16.27  Citi  15.40  err +0.87bp
GOLDS: ours  22.46  Citi  22.29  err +0.17bp



negative control — annual matched swap: 12.81bp (gap -3.46bp vs Q/Q)
OK: the annual convention fails by the documented ~4bp, as it must.


### 3.2 The Blues ticket, and the sizing rule

Pure arithmetic against Citi's published marks (09-Feb → 06-Jun-2017):
the CA leg's gross is exact, and the β-sizing rule reproduces Citi's
published belly notional to 0.4%.

In [5]:
ca_leg = (8.8 - 6.6) * 200_000.0
assert np.isclose(ca_leg, 440_000.0)
belly_dv01 = 0.214 * 200_000.0            # β=21.4 bp-per-percent ≡ 0.214 bp/bp
CITI_BELLY = 41_088.0                     # published 5y notional × era $/bp
print(f"CA leg gross  (8.8−6.6)×$200k = ${ca_leg:,.0f}   -> matches Citi exactly")
print(f"belly sizing  0.214×$200k = ${belly_dv01:,.0f}/bp vs Citi ${CITI_BELLY:,.0f} "
      f"(ratio {belly_dv01 / CITI_BELLY:.3f})")
assert 0.99 < belly_dv01 / CITI_BELLY < 1.05
hedge_leg = belly_dv01 * (-16.5 - -18.2)
print(f"fly −18.2 → −16.5: paid belly earns ${hedge_leg:+,.0f}; "
      f"reconstruction ${ca_leg + hedge_leg:,.0f} vs Citi net +$500,000 "
      f"({abs(ca_leg + hedge_leg - 500_000) / 500_000:.1%} off)")
assert abs(ca_leg + hedge_leg - 500_000.0) / 500_000.0 < 0.03

CA leg gross  (8.8−6.6)×$200k = $440,000   -> matches Citi exactly
belly sizing  0.214×$200k = $42,800/bp vs Citi $41,088 (ratio 1.042)
fly −18.2 → −16.5: paid belly earns $+72,760; reconstruction $512,760 vs Citi net +$500,000 (2.6% off)


## 4. Sign probes, live

### 4.1 The engine, on a five-mark micro-book

A short-spread BLUES × 2s5s10s book (buy the pack, pay the swap, pay the
belly) and its exact mirror, run through `QueryDrivenBacktest` on five
marks. The two books must be equal and opposite to the dollar, and the short
book's P&L must track −ΔCA to the sign.

In [6]:
from RVUtils.ConvexityRV import cavf_engine as E

# A probe window must carry a REAL CA move, or the convention residuals
# dominate: the engine's traded swap is the annual-spec instrument, whose
# rate moves 0.75·r·Δr MORE than the Q/Q matched rate per move (measured
# +0.58bp on the 20bp rally week of 2024-07-01..09 — exactly the compounding
# term, the same residual w2b recorded as its 0.90 slope). The window below
# is the March-2022 dislocation: BLUES CA fell ~9bp in eight sessions, no
# IMM roll inside it.
_probe_days = [d for d in ca_spl["BLUES"].index
               if dt.date(2022, 3, 29) <= d.date() <= dt.date(2022, 4, 8)]
_specs = {}
for side in (-1, +1):
    _specs[side] = E.spec_from_episode(
        label="BLUES", side=side, entry=_probe_days[0].date(),
        exit=_probe_days[-2].date(), beta_entry=0.2, fly=FLIES["2s5s10s"])
_bt = {side: E.run_backtest([sp], _probe_days) for side, sp in _specs.items()}
eqS = E.assert_ran(_bt[-1], [_specs[-1]], expect_days=len(_probe_days))
eqL = E.assert_ran(_bt[+1], [_specs[+1]], expect_days=len(_probe_days))
mirror = float((eqS + eqL).abs().max())
print(f"short-book terminal ${float(eqS.iloc[-1]):+,.0f}   "
      f"long-book ${float(eqL.iloc[-1]):+,.0f}   max |sum| ${mirror:,.0f}")
assert mirror < 0.01 * max(1.0, float(eqS.abs().max())), (
    "the long and short books are not mirrors — a sign is being dropped")

d_ca = float(ca_raw["BLUES"].loc[_probe_days[-2]] - ca_raw["BLUES"].loc[_probe_days[0]])
d_fly = float(fly_by_id["2s5s10s"].loc[_probe_days[-2]] - fly_by_id["2s5s10s"].loc[_probe_days[0]])
pred = -1 * (d_ca - 0.2 * d_fly) * CFG.ca_dv01
print(f"ΔCA {d_ca:+.2f}bp  Δfly {d_fly:+.2f}bp  panel-predicted short-book "
      f"${pred:+,.0f} vs engine ${float(eqS.iloc[-1]):+,.0f}")
assert abs(d_ca) > 3.0, "the probe window no longer carries a real CA move"
assert np.sign(pred) == np.sign(float(eqS.iloc[-1])), (
    "the engine book and the panel arithmetic disagree on SIGN")
print("OK: short spread = buy pack + pay swap + pay belly; the mirror is")
print("exact, and on a window with a real CA move the engine tracks the")
print("panel's sign and order of magnitude.")

C:\Users\chris\anaconda3\envs\stir\Lib\site-packages\rateslib\data\fixings.py:3426: RuntimeWarning:

invalid value encountered in divide



short-book terminal $+488,381   long-book $-488,381   max |sum| $0
ΔCA -4.00bp  Δfly +1.05bp  panel-predicted short-book $+420,733 vs engine $+488,381
OK: short spread = buy pack + pay swap + pay belly; the mirror is
exact, and on a window with a real CA move the engine tracks the
panel's sign and order of magnitude.


### 4.2 The execution convention, on pure noise

The known answer is engineered: i.i.d. noise around a constant has no
tradeable content, yet same-day fills 'earn' it. This is the probe form of
the test that guards amendment 1.

In [7]:
_rng = np.random.default_rng(2024)
_idx = pd.bdate_range("2021-01-04", periods=900)
_noise = pd.Series(5.0 + _rng.normal(0, 1.0, len(_idx)), index=_idx)
_spec = G.CellSpec("probe", "A_ca", "ca_only", "X", None,
                   S.SignalConfig(window=252))
_g0 = float(G.run_cell(_spec, {"X": _noise}, {}, exec_lag_bd=0)
            .equity_by_mult[0.0].iloc[-1])
_g1 = float(G.run_cell(_spec, {"X": _noise}, {}, exec_lag_bd=1)
            .equity_by_mult[0.0].iloc[-1])
print(f"pure noise: same-day fills ${_g0:,.0f} vs t+1 fills ${_g1:,.0f}")
assert _g0 > 2_000_000 and _g1 < 0.15 * _g0
print("OK: the convention kills the phantom; anything left at t+1 is candidate real.")

pure noise: same-day fills $8,194,450 vs t+1 fills $741,726
OK: the convention kills the phantom; anything left at t+1 is candidate real.


## 5. The grid — 515 pre-registered cells

14 structures (5 packs, 4 CME bundles, 5 outright ranks) × 6 flies each
(three shapes, spot + matched forward start) × two threshold sets across
five signal families, plus CA-only / fly-only controls, the pinned
Citi-Blues fair-value line, the JPM beta-stability variant, and 30
positioning / CCP-basis / carry overlay cells.

In [8]:
stats = pd.read_parquet(DATA / "cavf_grid_stats.parquet")
rets = pd.read_parquet(DATA / "cavf_grid_returns.parquet")
eps = pd.read_parquet(DATA / "cavf_grid_episodes.parquet")
assert len(stats) == CFG.declared_trials
traded = stats[stats["n_ep"] > 0]
print(f"{len(traded)}/{len(stats)} cells traded; "
      f"{len(eps):,} episodes; median hold "
      f"{float(traded['mean_hold_bd'].median()):.0f}bd")
print("\nexit reasons:")
print(eps["exit_reason"].value_counts().to_string())

475/515 cells traded; 6,182 episodes; median hold 40bd

exit reasons:
exit_reason
z_exit         3450
max_hold       2540
end_of_data     158
z_stop           34


In [9]:
SHOW = ["family", "structure", "fly", "z_in", "n_ep", "hit", "gross_usd",
        "net_1x", "ann_sharpe", "n_eff"]
print("top 12 by annualised Sharpe (zero cost, t+1 fills, roll-spliced):")
print(stats.nlargest(12, "ann_sharpe")[SHOW].round(3).to_string())
print("\nfamily medians:")
FAM = stats.groupby("family")[["n_ep", "gross_usd", "net_0p5x", "net_1x",
                               "net_2x", "ann_sharpe"]].median().round(0)
print(FAM.to_string())

# every CA-based family's median gross is negative; the sole positive-median
# family is the fly-only CONTROL — asserted so the prose cannot drift from the
# artifact
for fam in ("A", "A_ca", "B"):
    assert float(FAM.loc[fam, "gross_usd"]) < 0, (
        f"family {fam} median gross turned positive — §9's verdict text is stale")
assert float(FAM.loc[A_FLY := "A_fly", "gross_usd"]) > 0
assert float(FAM.loc["A_fly", "net_1x"]) < 0, (
    "the fly-only control now survives its own costs — re-derive the verdict")

top 12 by annualised Sharpe (zero cost, t+1 fills, roll-spliced):
                       family structure         fly  z_in  n_ep    hit    gross_usd       net_1x  ann_sharpe    n_eff
cell_id                                                                                                              
Afly|2s5s10s@5Y|s       A_fly      None  2s5s10s@5Y   1.5    34  0.794  2463518.973  -936481.027       1.232  101.273
Afly|2s5s10s@5Y|p       A_fly      None  2s5s10s@5Y   2.0    18  0.944  1601369.019  -198630.981       1.094   99.691
Afly|1s2s3s@4Y|s        A_fly      None   1s2s3s@4Y   1.5    45  0.778  1486990.345 -3013009.655       1.067  180.232
Afly|1s2s3s@4Y|p        A_fly      None   1s2s3s@4Y   2.0    18  0.889   941728.571  -858271.429       0.914  131.551
Afly|2s3s5s@2Y|s        A_fly      None   2s3s5s@2Y   1.5    29  0.759  1716331.170 -1183668.830       0.866   93.447
Afly|2s3s5s@4Y|s        A_fly      None   2s3s5s@4Y   1.5    23  0.913   756310.464 -1543689.536       0.843

### 5.1 What the two amendments were worth — the artifact ledger

The same-day-fill diagnostic and the roll splice each removed a *specific,
measured* phantom. This section is the receipt.

In [10]:
diag = META["same_day_diag"]
print(f"median hit rate: primary {diag['median_hit_primary']:.3f} vs "
      f"same-day fills {diag['median_hit_same_day']:.3f}")
print("\nmedian mark-noise harvest by family (same-day gross − t+1 gross, USD):")
print(pd.Series(diag["median_noise_harvest_by_family"]).round(0).to_string())
print(f"\nsplice magnitude (cumulative |raw − spliced|, bp): "
      f"{ {k: round(v, 1) for k, v in META['splice_max_gap_bp'].items()} }")
print("\nThe first pass — before either amendment — printed hit rates of 1.000")
print("and per-episode P&L an order of magnitude above the CA level itself.")
print("Neither phantom is tradeable: one is the mark's own measurement noise,")
print("the other is the IMM-roll label switch (22 of 33 rolls are FOMC dates).")

median hit rate: primary 0.455 vs same-day fills 0.542

median mark-noise harvest by family (same-day gross − t+1 gross, USD):
A           615352.0
A_ca        842863.0
A_fly       411053.0
B           762741.0
B_jpm            0.0
B_pinned    758842.0
C_posit     531393.0
D_basis     217421.0
E_carry     148153.0

splice magnitude (cumulative |raw − spliced|, bp): {'WHITES': 24.9, 'GOLDS': 26.9, 'SFR12': 93.7}

The first pass — before either amendment — printed hit rates of 1.000
and per-episode P&L an order of magnitude above the CA level itself.
Neither phantom is tradeable: one is the mark's own measurement noise,
the other is the IMM-roll label switch (22 of 33 rolls are FOMC dates).


## 6. The null, the deflated Sharpe, and the placebo

Both clocks, per the house discipline: per-hold (n_eff from holding periods)
and annualised (1/√span). The deflated Sharpe of the best cell uses the
actual cross-cell correlation (Bailey's effective-N, the direction that makes
the test harsher).

In [11]:
from RVUtils.StatisticalFinance.deflated_sharpe import (
    deflated_sharpe_of_best, expected_max_sharpe)

med_neff = float(traded["n_eff"].median())
span_y = float(traded["span_y"].median())
bars = {
    "per-hold": expected_max_sharpe(CFG.declared_trials, 1.0 / med_neff),
    "annualised": expected_max_sharpe(CFG.declared_trials, 1.0 / span_y),
}
best = stats["ann_sharpe"].idxmax()
best_sr = float(stats.loc[best, "ann_sharpe"])
print(f"median n_eff {med_neff:.1f}   span {span_y:.2f}y")
print(f"E[max SR | null, {CFG.declared_trials} trials]: "
      f"per-hold {bars['per-hold']:.3f}   annualised {bars['annualised']:.3f}")
print(f"best cell: {best}  ann Sharpe {best_sr:.3f}")
assert best_sr < bars["annualised"], (
    "the best cell now clears the annualised null — re-derive the verdict")

trial_cols = [c for c in rets.columns if rets[c].abs().sum() > 0]
dsr = deflated_sharpe_of_best([rets[c].to_numpy() for c in trial_cols])
print(f"\ndeflated Sharpe of the best of {len(trial_cols)} traded cells "
      f"(effective N {dsr['n_trials_effective']:.0f}, method "
      f"{dsr['effective_n_method']}): DSR {dsr['dsr']:.3f}")
assert dsr["dsr"] < 0.95, (
    "the winner clears DSR 0.95 — the verdict below is stale")

print("\nplacebo (signal lagged +20bd) on the top cells:")
for k, v in META["placebo_lag20"].items():
    print(f"  {k}: live ${v['gross_live']:,.0f} -> lag20 ${v['gross_lag20']:,.0f}")

median n_eff 35.8   span 5.63y
E[max SR | null, 515 trials]: per-hold 0.512   annualised 1.291
best cell: Afly|2s5s10s@5Y|s  ann Sharpe 1.232



deflated Sharpe of the best of 475 traded cells (effective N 447, method bailey): DSR 0.223

placebo (signal lagged +20bd) on the top cells:
  Bj|BUNDLE3Y|2s5s10s@2Y: live $215,260 -> lag20 $40,134
  A|SFR20|1s2s3s@5Y|p: live $2,230,811 -> lag20 $-188,408
  Bj|SFR16|2s3s5s: live $847,359 -> lag20 $0


## 7. The overlays — positioning, CME–LCH basis, carry

Declared as conditioning overlays with pre-registered weak expectations
(the positioning mechanism reproduces on Blues only; the basis mechanism is
IM non-nettability, not the level). They gate entries of the pack ×
2s5s10s books.

In [12]:
ov = stats[stats["family"].isin(["C_posit", "D_basis", "E_carry"])]
base = stats.loc[[c for c in stats.index
                  if any(c == o.split("|", 1)[1] for o in ov.index)]]
print("overlay cells:")
print(ov[SHOW].round(3).to_string())
print("\ntheir base books:")
print(base[["family", "structure", "n_ep", "gross_usd", "ann_sharpe"]]
      .round(3).to_string())
print("\nOverlays mostly cut episode counts to single digits without turning")
print("a negative-median family positive — conditioning cannot rescue a")
print("spread that does not revert tradably in the first place.")

overlay cells:


                       family structure      fly  z_in  n_ep    hit    gross_usd       net_1x  ann_sharpe    n_eff
cell_id                                                                                                           
C|A|WHITES|2s5s10s|p  C_posit    WHITES  2s5s10s   2.0     5  0.800  1346003.244   937756.397       0.212   39.166
D|A|WHITES|2s5s10s|p  D_basis    WHITES  2s5s10s   2.0     2  0.500    10458.472  -144587.288       0.046  567.129
E|A|WHITES|2s5s10s|p  E_carry    WHITES  2s5s10s   2.0     0    NaN        0.000        0.000         NaN      NaN
C|B|WHITES|2s5s10s|p  C_posit    WHITES  2s5s10s   2.0     5  0.600  -800609.363 -1223549.351      -0.182   90.886
D|B|WHITES|2s5s10s|p  D_basis    WHITES  2s5s10s   2.0     2  1.000    73896.901   -84985.943       0.474  708.912
E|B|WHITES|2s5s10s|p  E_carry    WHITES  2s5s10s   2.0     0    NaN        0.000        0.000         NaN      NaN
C|A|REDS|2s5s10s|p    C_posit      REDS  2s5s10s   2.0     4  0.500   324687.460

## 8. The headline measurement: does the matched-start fly hedge better?

The forward-start hypothesis — a fly starting at the structure's expiry
should co-move with its CA more than the spot fly — was previously REJECTED,
but only on ranks ≤10 (T1 ≤ 2.5y); Blues and Golds were out of reach. This
is the first measurement at full depth, **and it changes the reading**: the
front structures peak at SPOT and die by a 2Y start, but the deep books'
peak-R² start now RISES with depth (Blues peaks at 2Y, Golds at 3Y — the
slope below is +0.69 against the shallow-era −0.018). The mechanism became
visible once the deep data existed. What did NOT appear is strength: no cell
of the matrix reaches an R² that would size a hedge, and the grid cells
built on matched-start flies still lose (§5). A mechanism without a trade.


In [13]:
fsm = pd.read_parquet(DATA / "cavf_fs_matrix.parquet")
best_per = (fsm.loc[fsm.groupby(["structure", "start_y"])["r2"].idxmax()]
            .pivot(index="structure", columns="start_y", values="r2"))
order = [s.label for s in U.STRUCTURES if s.label in best_per.index]
best_per = best_per.loc[order]
print("hedge R² of ΔCA on Δfly (best of 3 shapes), structure × forward start:")
print(best_per.round(3).to_string())

peaks = []
for s in U.STRUCTURES:
    sub = fsm[fsm["structure"] == s.label]
    if sub.empty:
        continue
    by_start = sub.groupby("start_y")["r2"].max()
    peaks.append({"structure": s.label, "t1": s.t1_mean_y,
                  "peak_start": float(by_start.idxmax()),
                  "peak_r2": float(by_start.max())})
PK = pd.DataFrame(peaks)
slope = np.polyfit(PK["t1"], PK["peak_start"], 1)[0]
print(f"\npeak-R² forward start regressed on structure T1: slope {slope:+.3f} "
      f"(the hypothesis predicts +1.0; the shallow-only rejection measured "
      f"−0.018)")
print(f"max R² anywhere in the matrix: {fsm['r2'].max():.3f}")

fig = go.Figure(go.Heatmap(
    z=best_per.to_numpy(), x=[f"{c:.0f}Y" for c in best_per.columns],
    y=best_per.index, colorscale="Viridis", zmin=0,
    colorbar={"title": "R²"}))
fig.update_layout(title="Hedge R² of ΔCA on Δfly — structure × fly forward start",
                  height=430)
fig.show()

hedge R² of ΔCA on Δfly (best of 3 shapes), structure × forward start:
start_y      0.0    1.0    2.0    3.0    4.0    5.0
structure                                          
WHITES     0.386  0.125  0.014  0.004  0.003  0.003
REDS       0.321  0.114  0.005  0.003  0.002  0.001
GREENS     0.025  0.040  0.018  0.020  0.004  0.010
BLUES      0.181  0.229  0.332  0.148  0.065  0.011
GOLDS      0.179  0.168  0.200  0.240  0.041  0.007
BUNDLE2Y   0.094  0.032  0.049  0.025  0.014  0.010
BUNDLE3Y   0.067  0.079  0.023  0.019  0.011  0.007
BUNDLE4Y   0.139  0.181  0.195  0.093  0.040  0.017
BUNDLE5Y   0.133  0.139  0.133  0.161  0.053  0.021
SFR4       0.336  0.117  0.016  0.002  0.002  0.002
SFR8       0.247  0.079  0.002  0.005  0.006  0.004
SFR12      0.061  0.114  0.072  0.042  0.018  0.014
SFR16      0.148  0.160  0.134  0.179  0.056  0.009
SFR20      0.045  0.049  0.084  0.054  0.027  0.006



peak-R² forward start regressed on structure T1: slope +0.692 (the hypothesis predicts +1.0; the shallow-only rejection measured −0.018)
max R² anywhere in the matrix: 0.386


## 9. Engine certification — decomposed, because the blend hides the answer

Representative books ran end-to-end through `QueryDrivenBacktest` — real
futures legs, a date-pinned matched swap, a date-pinned fly — on the panel's
own fill dates. Three effects separate engine from panel and each is shown
on its own: (a) the **CA package itself** (the `__nofly` variant, non-roll
episodes) is the hard gate; (b) episodes **crossing an IMM roll** differ by
construction (the engine holds the original contracts, the panel rolls at
zero cost); (c) the **fly leg ages** in the engine while the panel's is
constant-maturity — the same drift that kept only 32–49% of panel dollars
in the previous grid's hedged cells.

The certification also caught two live defects on its first pass (corr
−0.005): negative `contracts` on a `STIRFutureQuery` silently going LONG
(the builder's weight flip cancels against the leg's own sign), and an
`abs()` on the pair β that flipped the fly leg whenever β < 0. Both are
pinned by tests now; the numbers below are the post-fix state.

In [14]:
cert = json.loads((DATA / "cavf_certification.json").read_text())
CERT = pd.DataFrame(cert).T
print(CERT.to_string())

_gate = cert.get("A|BLUES|2s5s10s|p__nofly", {})
_g = _gate.get("corr_daily_nonroll_median", np.nan)
print(f"\nHARD GATE — CA package alone, non-roll episodes, median per-episode "
      f"daily corr: {_g:+.4f}")
assert np.isfinite(_g) and _g > 0.80, (
    f"the CA package does not certify ({_g}) — the panel's CA leg is not "
    "describing the tradeable book and every number above is suspect")
# 0.80, not strat2's 0.997, and the gap has a NAME: the TB path rounds the
# pack price to the ¼ tick before differencing (round_pack_to_tick=True), so
# the panel's daily change carries up to 0.125bp of quantisation against a
# daily sd of a few tenths — a corr ceiling well below 1 that the engine's
# unrounded settle marks do not share.
for cell, row in cert.items():
    if cell.endswith("__nofly") or not row.get("n_episodes", 0):
        continue
    print(f"{cell}: blended corr {row['corr_daily']:+.3f}, non-roll median "
          f"{row['corr_daily_nonroll_median']:+.3f}, engine "
          f"${row['engine_terminal']:,.0f} vs panel ${row['panel_terminal']:,.0f}")
print("""
Three named residuals separate engine dollars from panel dollars, and each is
measured, not waved at:
  1. TICK QUANTISATION (corr): the panel's pack price is ¼-tick rounded.
  2. ROLL HANDLING: 7/13 Blues episodes cross an IMM roll; the engine holds
     the original window, the panel rolls at zero cost.
  3. FIXED-WINDOW CARRY (terminals): a HELD window's CA slides down the T1²
     curve; a constant-rank panel structurally cannot see that slide. It is
     Citi's own '3m roll' column, not a discovery — and the engine-level book
     that DOES earn it was already measured by w2b on this same repaired
     data: gross Sharpe 0.130, below its own six-trial null. The carry does
     not rescue the family; it was already inside the corpse.
Any POSITIVE panel cell would still have to be re-derived through the engine
before being believed, per inheritance ban #7. None qualified.""")

                          n_episodes  n_crossing_rolls  n_marks  corr_daily  corr_daily_nonroll_median  engine_terminal  panel_terminal  active_days  seconds
A|BLUES|2s5s10s|p               13.0               7.0   1130.0    0.344117                   0.901044     2.865882e+06    9.266324e+05        479.0     59.0
A|GOLDS|2s5s10s@5Y|p            11.0               9.0   1019.0    0.391182                   0.380780     3.089315e+06    1.223901e+05        571.0    121.0
A|SFR20|1s2s3s@5Y|p              6.0               3.0    671.0    0.307407                   0.465066    -4.752046e+05    2.230811e+06        199.0    134.0
A|BLUES|2s5s10s|p__nofly        13.0               7.0   1130.0    0.359244                   0.850073     3.327808e+06    8.305724e+05        479.0    174.5

HARD GATE — CA package alone, non-roll episodes, median per-episode daily corr: +0.8501
A|BLUES|2s5s10s|p: blended corr +0.344, non-roll median +0.901, engine $2,865,882 vs panel $926,632
A|GOLDS|2s5s10s@5Y|p:

## 10. The books, visually

In [15]:
pick = {
    "best cell (fly-only control)": stats.nlargest(1, "ann_sharpe").index[0],
    "best CA cell": stats[~stats["family"].isin(["A_fly"])]
        .nlargest(1, "ann_sharpe").index[0],
    "Citi-pinned Blues FV": "Bpin|BLUES|2s5s10s",
    "Citi structure, pairs": "A|BLUES|2s5s10s|p",
}
fig = go.Figure()
for name, cid in pick.items():
    if cid in rets.columns:
        eq = rets[cid].cumsum()
        fig.add_trace(go.Scatter(x=eq.index, y=eq.to_numpy(), mode="lines",
                                 name=f"{name} [{cid}]"))
fig.add_hline(y=0, line_width=1, line_color="#888")
fig.update_layout(title="CA-vs-fly — representative books, zero cost, "
                        "t+1 fills, roll-spliced",
                  yaxis_title="cumulative P&L, USD", height=460,
                  legend={"orientation": "h", "y": -0.18})
fig.show()

In [16]:
from BT.trade_dashboard import trade_dashboard

_best_ca = pick["best CA cell"]
book = eps[eps.cell_id == _best_ca].copy()
if len(book):
    per_ep_ret = rets[_best_ca]
    rows = []
    for _, r in book.iterrows():
        idx2 = per_ep_ret.index
        w = per_ep_ret.loc[r["entry"]:r["exit"]]
        rows.append({"closed_at": r["exit"], "pnl": float(w.sum()),
                     "side": "short spread" if r["side"] < 0 else "long spread",
                     "structure": _best_ca, "z": float(r["z_at_entry"]),
                     "exit_reason": r["exit_reason"],
                     "holding_period_days": int(r["hold_bd"])})
    bdf = pd.DataFrame(rows)
    span = (bdf["closed_at"].max() - bdf["closed_at"].min()).days / 365.25
    fig = trade_dashboard(bdf, title=f"best CA cell — {_best_ca}",
                          span_years=max(span, 0.5), signal_col="z")
    fig.show()

## 11. Robustness on the best CA cell

In [17]:
eq_best = rets[_best_ca].cumsum()
d = rets[_best_ca]
by_year = d.resample("YE").sum()
print("P&L by year, best CA cell (zero cost):")
print(by_year.round(0).to_string())
loo = {}
for y in sorted(set(d.index.year)):
    keep = d[d.index.year != y]
    sd = keep.std(ddof=1)
    loo[y] = {"terminal_ex": float(keep.sum()),
              "sharpe_ex": float(keep.mean() / sd * math.sqrt(252)) if sd > 0 else np.nan}
print("\nleave-one-year-out:")
print(pd.DataFrame(loo).T.round(2).to_string())

F = pd.DataFrame({
    "level": wide[["2Y", "5Y", "10Y"]].mean(axis=1) * 100.0,
    "slope": (wide["10Y"] - wide["2Y"]) * 100.0,
    "curv": (2 * wide["5Y"] - wide["2Y"] - wide["10Y"]) * 100.0}).diff()
F["level_sq"] = F["level"] ** 2
A = pd.concat([d.rename("pnl"), F], axis=1).dropna()
A = A[A["pnl"] != 0]
if len(A) > 30:
    X = sm.add_constant(A[["level", "slope", "curv", "level_sq"]].to_numpy())
    fit = sm.OLS(A["pnl"].to_numpy(), X).fit(cov_type="HAC",
                                             cov_kwds={"maxlags": 5})
    print(f"\nfactor attribution ({len(A)} non-flat days): "
          f"R² {float(fit.rsquared):.4f}; t(level) {float(fit.tvalues[1]):+.2f} "
          f"t(slope) {float(fit.tvalues[2]):+.2f} "
          f"t(curv) {float(fit.tvalues[3]):+.2f} "
          f"t(level²) {float(fit.tvalues[4]):+.2f}")
    print("A P&L this size with no curve loading is residual noise, not a")
    print("factor bet — consistent with the spread never reverting tradably.")

P&L by year, best CA cell (zero cost):
2021-12-31         0.0
2022-12-31    215260.0
2023-12-31         0.0
2024-12-31         0.0
2025-12-31         0.0
2026-12-31         0.0
Freq: YE-DEC

leave-one-year-out:
      terminal_ex  sharpe_ex
2021     215260.5       0.70
2022          0.0        NaN
2023     215260.5       0.70
2024     215260.5       0.70
2025     215260.5       0.70
2026     215260.5       0.67

factor attribution (61 non-flat days): R² 0.0950; t(level) +0.11 t(slope) +0.07 t(curv) +1.33 t(level²) +0.26
A P&L this size with no curve loading is residual noise, not a
factor bet — consistent with the spread never reverting tradably.


## 12. Reading this notebook

* **The family is dead, measured honestly at its own declared size.** Across
  515 pre-registered cells on repaired data: every CA-based family (pairs,
  fair-value, CA-only, JPM-threshold, the pinned Citi line) has a NEGATIVE
  median gross before a basis point of cost; the best single cell fails the
  annualised E[max SR | null] and the deflated Sharpe; the placebo kills the
  top cells; the overlays (positioning, CME–LCH basis, carry) thin the books
  without changing the sign.

* **The two most valuable numbers in the block are the phantoms.** Same-day
  fills add ~$0.4–0.9M of un-tradeable "profit" per median cell (hit rates
  0.93 vs 0.45 real); the IMM-roll label switch adds signed jumps 22 of 33
  of which land on FOMC dates. Any CA mean-reversion result that does not
  state its fill convention and roll handling should be assumed to be
  harvesting one or both.

* **The forward-start hypothesis: mechanism yes, trade no — a genuine
  update.** With Blues and Golds finally daily 2021–2026, the peak-R²
  forward start rises with the structure's depth (slope +0.69 vs the
  shallow-only −0.018 that grounded the old rejection) — the "vol at the
  pack's own expiry" mechanism is visible for the first time. But its
  strength tops out at R² 0.33 on the deep books (0.386 anywhere, at the
  front-spot corner): too weak to size a hedge, and the declared cells
  built on matched-start flies lose like everything else. Citi's own
  2s5s10s remains a fair-value regressor at monthly horizons, not a daily
  hedge.

* **What survives is measurement, not a trade.** The CA panel itself (31
  structures, zero failures, tied to Citi's printed screen at ~1bp with the
  annual convention failing by −4bp as a negative control), the fly-leg
  panel, the certification path, and the screener these feed.

In [18]:
summary = {
    "declared_trials": CFG.declared_trials,
    "cells_traded": int(len(traded)),
    "best_cell": best,
    "best_ann_sharpe": round(best_sr, 3),
    "null_annualised": round(bars["annualised"], 3),
    "null_perhold": round(bars["per-hold"], 3),
    "dsr_best": round(float(dsr["dsr"]), 3),
    "median_gross_A": float(FAM.loc["A", "gross_usd"]),
    "median_gross_B": float(FAM.loc["B", "gross_usd"]),
    "median_gross_A_ca": float(FAM.loc["A_ca", "gross_usd"]),
    "median_hit_primary": round(diag["median_hit_primary"], 3),
    "median_hit_same_day": round(diag["median_hit_same_day"], 3),
    "fs_hypothesis_slope": round(float(slope), 3),
    "fs_hypothesis_max_r2": round(float(fsm["r2"].max()), 3),
}
for k, v in summary.items():
    print(f"{k:24} {v}")
pd.Series(summary).to_csv(DATA / "cavf_backtest_summary.csv")
print(f"\nwrote {DATA / 'cavf_backtest_summary.csv'}")

declared_trials          515
cells_traded             475
best_cell                Afly|2s5s10s@5Y|s
best_ann_sharpe          1.232
null_annualised          1.291
null_perhold             0.512
dsr_best                 0.223
median_gross_A           -591900.0
median_gross_B           -701497.0
median_gross_A_ca        -514426.0
median_hit_primary       0.455
median_hit_same_day      0.542
fs_hypothesis_slope      0.692
fs_hypothesis_max_r2     0.386

wrote C:\Users\chris\clee\ARBS-cvx3\notebooks\data\convexity_rv\cavf_backtest_summary.csv
